[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap08_finetuning_intro.ipynb)

# Capítulo 8 — Introdução ao Fine-Tuning de LLMs

> **Baseado no projeto [AI-Orchestrator](https://github.com/aejepsen/AI-Orchestrator) de Anderson Ejepsen**

Neste notebook, vamos entender a diferença fundamental entre **full fine-tuning** e **LoRA** (Low-Rank Adaptation), e por que LoRA é a abordagem viável para especializar modelos de 7-9B de parâmetros em hardware acessível como uma RTX 3060 ou Google Colab.

## 1. Instalação das dependências

Instalamos `transformers`, `peft` (Parameter-Efficient Fine-Tuning) e `bitsandbytes` para quantização. Essas são as bibliotecas fundamentais para qualquer workflow de fine-tuning com LoRA.

In [ ]:
# Instalação das bibliotecas necessárias
!pip install -q transformers peft accelerate bitsandbytes torch sentencepiece protobuf

## 2. O que é Fine-Tuning?

Fine-tuning é o processo de continuar o treinamento de um modelo pré-treinado em dados específicos do seu domínio. Existem duas abordagens principais:

### Full Fine-Tuning
- Atualiza **todos** os parâmetros do modelo
- Requer VRAM proporcional ao tamanho total do modelo (ex: ~36 GB para 9B em bf16)
- Risco maior de catastrophic forgetting
- Gera um modelo completo como artefato

### LoRA (Low-Rank Adaptation)
- Congela o modelo base e injeta **matrizes de baixo rank** nas camadas de atenção
- Treina tipicamente **0.5-2%** dos parâmetros originais
- VRAM drasticamente menor (~22 GB para 9B com LoRA bf16)
- Artefato pequeno (adapter de ~50-200 MB) que se soma ao modelo base
- Permite múltiplas especializações sobre a mesma base

No projeto AI-Orchestrator, usamos **LoRA bf16** (não QLoRA 4-bit) no Qwen3.5-9B, conforme recomendação da Unsloth para essa arquitetura.

## 3. Visualizando a diferença: Full Fine-Tuning vs LoRA

Vamos criar uma visualização que mostra a proporção de parâmetros treináveis em cada abordagem.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Dados reais do projeto AI-Orchestrator (Qwen3.5-9B)
total_params = 9_000_000_000  # ~9B parâmetros
lora_params = 54_000_000      # ~54M parâmetros treináveis com r=16

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full Fine-Tuning
axes[0].barh(['Treináveis', 'Total'], [total_params, total_params],
             color=['#e74c3c', '#2c3e50'], height=0.5)
axes[0].set_title('Full Fine-Tuning', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Parâmetros')
axes[0].text(total_params * 0.5, 0, f'{total_params/1e9:.1f}B (100%)',
             ha='center', va='center', color='white', fontweight='bold')

# LoRA
axes[1].barh(['Treináveis', 'Congelados'],
             [lora_params, total_params - lora_params],
             color=['#e74c3c', '#95a5a6'], height=0.5)
axes[1].set_title('LoRA (r=16, alpha=32)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Parâmetros')
pct = lora_params / total_params * 100
axes[1].text(lora_params * 0.5, 0, f'{lora_params/1e6:.0f}M ({pct:.1f}%)',
             ha='center', va='center', color='white', fontsize=9)

plt.suptitle('Parâmetros Treináveis: Full FT vs LoRA (Qwen3.5-9B)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nResumo:')
print(f'  Total de parâmetros:        {total_params/1e9:.1f}B')
print(f'  Parâmetros LoRA treináveis: {lora_params/1e6:.0f}M ({pct:.2f}%)')
print(f'  Redução de VRAM:           ~36 GB -> ~22 GB (bf16 LoRA)')

## 4. Como funciona LoRA internamente

LoRA decompoe a atualização de peso `ΔW` em duas matrizes de rank baixo:

$$W' = W + \Delta W = W + B \times A$$

Onde:
- `W` é a matriz original (congelada, ex: 4096 x 4096)
- `A` tem dimensão `(r x 4096)` — rank baixo
- `B` tem dimensão `(4096 x r)` — rank baixo
- `r` é o rank (no AI-Orchestrator usamos `r=16`)
- `alpha` controla a escala da atualização (`alpha/r`)

Vamos visualizar isso.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.set_xlim(0, 16)
ax.set_ylim(0, 5)
ax.axis('off')

# Matriz W original (congelada)
w = patches.FancyBboxPatch((0.5, 1), 3, 3, boxstyle='round,pad=0.1',
                            facecolor='#95a5a6', edgecolor='#2c3e50', linewidth=2)
ax.add_patch(w)
ax.text(2, 2.5, 'W\n(congelada)\n4096 x 4096', ha='center', va='center',
        fontsize=11, fontweight='bold', color='white')

# Sinal de +
ax.text(4.3, 2.5, '+', fontsize=24, ha='center', va='center', fontweight='bold')

# Matriz B (treinável)
b = patches.FancyBboxPatch((5, 1), 0.8, 3, boxstyle='round,pad=0.1',
                            facecolor='#e74c3c', edgecolor='#c0392b', linewidth=2)
ax.add_patch(b)
ax.text(5.4, 2.5, 'B\n4096\nx r', ha='center', va='center',
        fontsize=9, fontweight='bold', color='white')

# Sinal de x
ax.text(6.3, 2.5, '×', fontsize=20, ha='center', va='center')

# Matriz A (treinável)
a = patches.FancyBboxPatch((6.8, 2), 3, 0.8, boxstyle='round,pad=0.1',
                            facecolor='#e74c3c', edgecolor='#c0392b', linewidth=2)
ax.add_patch(a)
ax.text(8.3, 2.4, 'A (r x 4096)', ha='center', va='center',
        fontsize=9, fontweight='bold', color='white')

# Sinal de =
ax.text(10.5, 2.5, '=', fontsize=24, ha='center', va='center', fontweight='bold')

# Resultado W'
wp = patches.FancyBboxPatch((11.2, 1), 3, 3, boxstyle='round,pad=0.1',
                             facecolor='#27ae60', edgecolor='#1e8449', linewidth=2)
ax.add_patch(wp)
ax.text(12.7, 2.5, "W'\n(atualizada)\n4096 x 4096", ha='center', va='center',
        fontsize=11, fontweight='bold', color='white')

ax.set_title('LoRA: W\' = W + B × A  (r=16, alpha=32)',
             fontsize=14, fontweight='bold', pad=20)

# Legenda
ax.text(2, 0.5, '■ Congelado', fontsize=10, color='#95a5a6')
ax.text(5.4, 0.5, '■ Treinável', fontsize=10, color='#e74c3c')
ax.text(8.3, 0.5, '■ Resultado', fontsize=10, color='#27ae60')

plt.tight_layout()
plt.show()

## 5. Carregando um modelo com PEFT/LoRA

Agora vamos carregar um modelo real e aplicar LoRA. Usaremos o `Qwen/Qwen2.5-1.5B` como exemplo (cabe em T4), mas o processo é idêntico para modelos maiores como o Qwen3.5-9B usado no AI-Orchestrator.

Os `target_modules` são as camadas onde LoRA injeta os adapters. No AI-Orchestrator, cobrimos:
- **Atenção**: `q_proj`, `k_proj`, `v_proj`, `o_proj`
- **MLP**: `gate_proj`, `up_proj`, `down_proj`

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# Modelo pequeno para demonstração (cabe em T4 16GB)
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

print("Carregando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Carregando modelo base...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Modelo carregado: {MODEL_NAME}")
print(f"Parâmetros totais: {sum(p.numel() for p in model.parameters()):,}")

## 6. Configurando e aplicando LoRA

A configuração LoRA define:
- **r** (rank): dimensão das matrizes de adaptação. Maior r = mais capacidade, mais VRAM
- **lora_alpha**: fator de escala. Geralmente `2*r` é um bom ponto de partida
- **lora_dropout**: regularização para evitar overfitting
- **target_modules**: quais camadas recebem adapters LoRA

In [ ]:
# Configuração LoRA idêntica à usada no AI-Orchestrator
lora_config = LoraConfig(
    r=16,                    # rank das matrizes A e B
    lora_alpha=32,           # fator de escala (alpha/r = 2.0)
    lora_dropout=0.05,       # dropout para regularização
    target_modules=[         # camadas que recebem adapters
        "q_proj", "k_proj", "v_proj", "o_proj",   # atenção
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
    bias="none",             # sem bias nos adapters
    task_type=TaskType.CAUSAL_LM,
)

# Aplicar LoRA ao modelo
model_lora = get_peft_model(model, lora_config)

# Exibir resumo de parâmetros
model_lora.print_trainable_parameters()

## 7. Visualizando parâmetros treináveis vs congelados

Vamos inspecionar cada camada do modelo para entender exatamente onde LoRA atua e quantos parâmetros estão sendo treinados em cada módulo.

In [ ]:
import pandas as pd

# Coletar informações de cada camada
dados = []
for name, param in model_lora.named_parameters():
    dados.append({
        'camada': name,
        'parametros': param.numel(),
        'treinavel': param.requires_grad,
        'dtype': str(param.dtype),
        'shape': str(list(param.shape)),
    })

df = pd.DataFrame(dados)

# Resumo por tipo
resumo = df.groupby('treinavel').agg(
    total_params=('parametros', 'sum'),
    num_camadas=('camada', 'count'),
).reset_index()
resumo['treinavel'] = resumo['treinavel'].map({True: 'Treinável (LoRA)', False: 'Congelado'})
resumo['percentual'] = (resumo['total_params'] / resumo['total_params'].sum() * 100).round(2)

print("=" * 60)
print("RESUMO DE PARÂMETROS")
print("=" * 60)
print(resumo.to_string(index=False))
print()

# Mostrar apenas camadas LoRA (treináveis)
lora_layers = df[df['treinavel']]
print(f"\nCamadas LoRA ({len(lora_layers)} camadas treináveis):")
print("-" * 60)
for _, row in lora_layers.iterrows():
    print(f"  {row['camada']:<60} {row['parametros']:>10,}  {row['shape']}")

## 8. Comparação de VRAM: Full FT vs LoRA

Vamos estimar e comparar o consumo de VRAM para diferentes abordagens de fine-tuning. Esses números são críticos para escolher o hardware certo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Estimativas de VRAM para Qwen3.5-9B
abordagens = {
    'Full FT\n(bf16)': {
        'modelo': 18,      # 9B * 2 bytes (bf16)
        'gradientes': 18,  # mesma VRAM que o modelo
        'otimizador': 36,  # AdamW: 2x modelo
        'ativacoes': 4,    # com gradient checkpointing
    },
    'LoRA bf16\n(AI-Orch)': {
        'modelo': 18,      # base congelada em bf16
        'gradientes': 0.1, # só adapters
        'otimizador': 0.4, # AdamW 8-bit dos adapters
        'ativacoes': 3,    # gradient checkpointing
    },
    'QLoRA\n(4-bit)': {
        'modelo': 5,       # base quantizada 4-bit
        'gradientes': 0.1, # adapters em bf16
        'otimizador': 0.4, # adapters
        'ativacoes': 2,    # menor contexto
    },
}

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(abordagens))
width = 0.5
bottom = np.zeros(len(abordagens))
cores = ['#2c3e50', '#e74c3c', '#f39c12', '#3498db']
labels = ['Modelo', 'Gradientes', 'Otimizador', 'Ativações']

for i, comp in enumerate(['modelo', 'gradientes', 'otimizador', 'ativacoes']):
    vals = [v[comp] for v in abordagens.values()]
    ax.bar(x, vals, width, bottom=bottom, label=labels[i], color=cores[i])
    bottom += vals

# Linhas de referência de hardware
ax.axhline(y=12, color='#e67e22', linestyle='--', alpha=0.7, label='RTX 3060 (12 GB)')
ax.axhline(y=16, color='#9b59b6', linestyle='--', alpha=0.7, label='T4 (16 GB)')
ax.axhline(y=24, color='#1abc9c', linestyle='--', alpha=0.7, label='L4 (24 GB)')
ax.axhline(y=40, color='#2ecc71', linestyle='--', alpha=0.7, label='A100 (40 GB)')

totals = [sum(v.values()) for v in abordagens.values()]
for i, t in enumerate(totals):
    ax.text(i, t + 1, f'{t:.1f} GB', ha='center', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(abordagens.keys())
ax.set_ylabel('VRAM (GB)')
ax.set_title('Consumo de VRAM por Abordagem (Qwen3.5-9B)', fontweight='bold')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, 85)
plt.tight_layout()
plt.show()

print("\nNo AI-Orchestrator, LoRA bf16 consumiu 31.8 GB de pico no A100 40GB.")
print("QLoRA 4-bit é contraindicado pela Unsloth para Qwen3.5 (arquitetura híbrida DeltaNet).")

## 9. Quando usar cada abordagem

| Critério | Full Fine-Tuning | LoRA | QLoRA |
|----------|-----------------|------|-------|
| VRAM necessária | ~76 GB (9B) | ~22 GB (9B) | ~8 GB (9B) |
| Qualidade | Melhor (teto) | Muito próxima | Leve degradação |
| Velocidade de treino | Lenta | Rápida | Rápida |
| Artefato | Modelo completo | Adapter (~100 MB) | Adapter (~100 MB) |
| Hardware mínimo (9B) | A100 80GB | A100 40GB / L4 | T4 16GB |
| Risco de overfit | Alto | Baixo | Baixo |

### Decisão no AI-Orchestrator

Escolhemos **LoRA bf16** porque:
1. Unsloth contraindica QLoRA em Qwen3.5 (camadas híbridas DeltaNet)
2. A100 40GB do Colab comporta confortavelmente
3. Resultado: routing 90.9%, injection 0/6 leaks, domains 87.5% — aprovado em todos os gates

## 10. Próximos passos

Agora que entendemos a teoria por trás do fine-tuning e LoRA, nos próximos capítulos vamos:

1. **Cap. 9 — Datasets**: Preparar dados no formato correto para SFT
2. **Cap. 10 — Treino LoRA**: Executar o treino completo com Unsloth
3. **Cap. 11 — Avaliação**: Medir a qualidade do modelo fine-tuned
4. **Cap. 12 — Export GGUF**: Converter para formato Ollama e deploy local